# Module 10 — RAG as a tool the model chooses

**THE ONE IDEA:** classic RAG always retrieves. **Agentic RAG makes retrieval a tool**,
so the model decides *whether* to retrieve, *what* to query, and *whether the result was
good enough to try again*.

```
CLASSIC   question -> retrieve -> generate          a WORKFLOW (same path every time)
AGENTIC   question -> model decides -> retrieve? -> reformulate? -> generate    an AGENT
```

You buy adaptivity. You pay in latency, cost, and evaluation difficulty — you are now
evaluating a **trajectory**, not a single retrieval.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from _providers import get_client
from _tools import _POLICY, openai_schemas, run_tool

client, MODEL, _ = get_client("openai")

# A deliberately naive retriever over the policy dict in _tools.py.
# Keyword overlap, no embeddings — the RETRIEVAL QUALITY is not the lesson here,
# WHO DECIDES TO CALL IT is.
def retrieve(query, k=2):
    q = set(query.lower().split())
    scored = sorted(_POLICY.items(),
                    key=lambda kv: len(q & set((kv[0] + " " + kv[1]).lower().split())),
                    reverse=True)
    return [f"[{k_}] {v}" for k_, v in scored[:k]]

print(retrieve("early repayment charge")[0])

## Classic RAG — retrieval is unconditional

One path. Retrieval happens even when the model already knows the answer, or when the
question has nothing to do with the corpus.

In [ ]:
def classic_rag(question):
    docs = retrieve(question)
    prompt = "Answer using ONLY this context.\n\n" + "\n".join(docs) + f"\n\nQ: {question}"
    r = client.chat.completions.create(model=MODEL, max_tokens=250,
                                       messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip(), 1     # always exactly 1 retrieval

for q in ["What is the early repayment charge in year 3?", "What is 17 times 4?"]:
    ans, n = classic_rag(q)
    print(f"\nQ: {q}\n  retrievals={n}  A: {ans[:110]}")
print("\n^ note the second one retrieved mortgage policy to do arithmetic.")

## Agentic RAG — retrieval is a tool

`search_policy` from `_tools.py` *is* the retriever, exposed as a schema. The model may
call it, call it twice with a better query, or skip it entirely.

In [ ]:
def agentic_rag(question, max_steps=5, verbose=True):
    messages = [{"role": "user", "content": question}]
    calls = []
    for _ in range(max_steps):
        r = client.chat.completions.create(
            model=MODEL, max_tokens=400,
            tools=openai_schemas(["search_policy", "calculate"]), messages=messages)
        msg = r.choices[0].message
        if r.choices[0].finish_reason != "tool_calls":
            return msg.content.strip(), calls
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            calls.append(f"{tc.function.name}({list(args.values())[0]!r})")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": run_tool(tc.function.name, args)})
    return None, calls

for q in ["What is the early repayment charge in year 3?", "What is 17 times 4?"]:
    ans, calls = agentic_rag(q)
    print(f"\nQ: {q}\n  tool calls: {calls}\n  A: {ans[:110]}")

## The lesson

In [ ]:
print("LESSON — the difference is not the retriever. It is WHO DECIDES.")
print()
print("  classic   retrieve ALWAYS         1 retrieval, every question, same path")
print("  agentic   retrieve IF USEFUL      0 for arithmetic, 1+ for policy, can retry")
print()
print("Classic RAG retrieved mortgage policy to answer '17 times 4'. It cannot not.")
print("Agentic RAG skipped the corpus and reached for calculate instead.")
print()
print("The price of that judgement:")
print("  - at least one extra LLM call per decision")
print("  - unbounded retrievals until you cap them (module 13)")
print("  - failure modes COMPOUND: bad retrieval poisons reasoning, and an agent")
print("    that retries retrieval multiplies your RAG bill")
print("  - you now evaluate a TRAJECTORY, not a single retrieval (module 36)")
print()
print("Interview framing: 'Classic RAG is a workflow - retrieve then generate, same")
print("path every time. Agentic RAG makes retrieval a tool the model chooses, so it")
print("can reformulate a weak query or skip retrieval entirely. You buy adaptivity")
print("and pay in latency, cost and evaluation difficulty.'")

---

**Next:** `11_agent_cost_and_tokens.ipynb` — what all this loops costs.